# Anonimizacja danych: od rozmowy do porównania znaczeń

**Warsztat w Google Colab • Python • język polski**

Przejdziesz przez cały proces: nagranie rozmowy → wgranie audio → transkrypcja ASR → diaryzacja → ocena CER → maskowanie danych → zapis tekstu → porównanie znaczeń zdań.

Po warsztacie potrafisz:
- otrzymać tekst i znaczniki czasu z faster-whisper oraz etykiety rozmówców z pyannote;
- porównać transkrypcję z ręcznym zapisem za pomocą CER;
- wykrywać telefon, e-mail i PESEL regexami oraz osoby i miejsca modelem NER;
- przygotować tekst z oznaczeniami typu `[OSOBA]` lub `[OSOBA_1]`;
- zapisać wynik do pliku UTF-8 i przedstawić ograniczenia metody;
- utworzyć embeddingi zdań i wyświetlić macierz podobieństwa oraz, dodatkowo, korelacji.

**Oznaczenia komórek:** **GOTOWY KOD** — uruchom i obejrzyj wynik; **SZABLON** — uzupełnij miejsca z `TODO`; **WŁASNY KOD** — samodzielnie zaimplementuj rozwiązanie. Miejsca do uzupełnienia oznaczono komentarzami TODO.

## 0. Otwórz notatnik w Colabie

1. Pobierz ten plik `.ipynb`. W [Google Colab](https://colab.research.google.com/) wybierz **Plik → Prześlij notatnik** i wskaż pobrany plik.
2. Zapisz własną kopię w Dysku Google. Uruchamiaj komórki po kolei przez `Shift+Enter`.
3. Domyślna transkrypcja działa na CPU. Do szybszej diaryzacji możesz wybrać **Środowisko wykonawcze → Zmień typ środowiska wykonawczego → GPU**.
4. Uruchom instalację poniżej. Jeżeli Colab poprosi o restart sesji po instalacji, zrestartuj ją i ponownie wykonaj importy.
5. Nagranie i pliki robocze pozostają w tymczasowym środowisku Colaba; pobierz potrzebny wynik przed zakończeniem sesji.

Notatnik jest samodzielny: nie wymaga klonowania repozytorium ani modułów z poprzednich warsztatów. Sekcję tekstową można wykonać bez audio, używając fikcyjnego tekstu.

**Materiał ćwiczeniowy:** nagraj krótką, odegraną rozmowę za zgodą uczestników. Używaj wymyślonych danych. Maskowanie i stałe pseudonimy nie gwarantują pełnej anonimizacji: treść i głos mogą nadal umożliwiać rozpoznanie osoby. Przed udostępnieniem wyniku obejrzyj go ręcznie.

In [ ]:
# GOTOWY KOD — instalacja w środowisku notatnika, jeden raz na sesję.
# Osobną instalację opcjonalnej diaryzacji znajdziesz w sekcji 2.
%pip -q install "faster-whisper>=1.2,<2" "jiwer>=4,<5" "transformers>=4.48,<5" "sentence-transformers>=5,<6", "pyannote.audio>=4,<5" sentencepiece numpy pandas matplotlib seaborn

In [ ]:
import re
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Audio, display

## 1. Nagraj rozmowę i wykonaj transkrypcję ASR

**Zadanie:** w parze nagraj 30–60 sekund rozmowy telefonem lub dyktafonem i zapisz plik WAV, MP3 albo M4A. Mówcie na zmianę, a potem dodajcie jedno krótkie nakładanie głosów, żeby zobaczyć ograniczenia diaryzacji.

Przykładowy scenariusz (postacie i kontakty są fikcyjne):
> A: Dzień dobry, nazywam się Anna Kowalska i mieszkam w Krakowie.
> B: Jestem Jan Nowak, przyjechałem z Warszawy. Jak mogę się skontaktować?
> A: Mój adres to anna.kowalska@example.org, a numer to plus czterdzieści osiem, sto dwadzieścia trzy, czterysta pięćdziesiąt sześć, siedemset osiemdziesiąt dziewięć.
> B: Dziękuję. Porozmawiajmy o tym, co pomaga odpocząć po pracy.

**ASR** (*Automatic Speech Recognition*) zamienia dźwięk na tekst. **Diaryzacja** odpowiada na pytanie „kto mówił kiedy?”, używając etykiet typu `SPEAKER_00`. Nie ustala nazwiska rozmówcy.

### Wgranie pliku — GOTOWY KOD
Poniższa komórka otworzy wybór pliku w Colabie. Wgraj jedno nagranie z komputera. Jego nazwa zostanie automatycznie zapisana w `PLIK_AUDIO`.

In [ ]:
# GOTOWY KOD — wgraj jedno nagranie z komputera.
from google.colab import files

przeslane = files.upload()
PLIK_AUDIO = next(iter(przeslane))

In [ ]:
# GOTOWY KOD — odsłuchaj wgrane nagranie przed transkrypcją.
display(Audio(filename=PLIK_AUDIO))

### Transkrypcja faster-whisper — SZABLON

Uzupełnij `MODEL_ASR` wartością `"small"` i `JEZYK_ASR` wartością `"pl"`. Mniejsze modele (`"base"`, `"tiny"`) są szybsze, ale ich wynik może być słabszy. Porównanie modeli ma sens dla tego samego nagrania i tej samej referencji.

- `beam_size=5` steruje przeszukiwaniem możliwych transkrypcji;
- `vad_filter=True` ogranicza przetwarzanie fragmentów bez mowy;
- `word_timestamps=True` daje czasy słów do połączenia z diaryzacją;
- pętla `for segment in generator` uruchamia przetwarzanie i zbiera wyniki transkrypcji.

Domyślnie używamy CPU i `int8`. GPU dla faster-whisper wymaga zgodnych bibliotek CUDA/cuDNN; jeżeli nie konfigurujesz ich na warsztacie, pozostań przy CPU. Pierwsze uruchomienie pobierze model.

Źródło: [dokumentacja faster-whisper](https://github.com/SYSTRAN/faster-whisper).

In [ ]:
# SZABLON — uzupełnij dwa parametry.
MODEL_ASR = None  # TODO: "small"
JEZYK_ASR = None  # TODO: "pl"

segmenty_asr = []
slowa_asr = []
transkrypcja = ""

# Uzupełnij MODEL_ASR i JEZYK_ASR przed uruchomieniem.
from faster_whisper import WhisperModel

model_asr = WhisperModel(MODEL_ASR, device="cpu", compute_type="int8")
generator, informacje = model_asr.transcribe(
    PLIK_AUDIO,
    language=JEZYK_ASR,
    beam_size=5,
    vad_filter=True,
    word_timestamps=True,
)
for segment in generator:
    segmenty_asr.append({
        "start": segment.start, "end": segment.end, "tekst": segment.text.strip()
    })
    for slowo in segment.words or []:
        slowa_asr.append({
            "start": slowo.start, "end": slowo.end, "tekst": slowo.word
        })
transkrypcja = " ".join(s["tekst"] for s in segmenty_asr)
print(transkrypcja)
# Jeśli wynik jest pusty, odsłuchaj nagranie i sprawdź, czy zawiera mowę.
display(pd.DataFrame(segmenty_asr).head(10))
del model_asr  # Dalej potrzebujemy tekstu i czasów, a nie modelu w pamięci.

## 2. Diaryzacja: przypisz wypowiedzi rozmówcom — SZABLON

Faster-whisper nie dostarcza samodzielnie diaryzacji. Użyjemy modelu [pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1) i połączymy etykiety ze znacznikami czasu słów.

Ta część jest opcjonalna i wymaga:
1. konta Hugging Face oraz zaakceptowania warunków na stronie modelu;
2. własnego [tokenu odczytu](https://huggingface.co/settings/tokens) z dostępem do tego modelu;
3. ustawienia `URUCHOM_DIARYZACJE = True` i uzupełnienia liczby rozmówców (dla tej rozmowy: `2`).

Token wpiszesz w ukrytym polu, nigdy w kodzie. Przy błędzie 401/403 sprawdź dostęp do modelu i uprawnienia tokenu. Bez dostępu pomiń tę część — regex, NER i porównanie zdań działają niezależnie.

Zainstaluj bibliotekę poniżej, zanim zaimportujesz pyannote. Jeśli instalacja wymaga restartu, po restarcie wykonaj wcześniejsze komórki ponownie. Pyannote 4 korzysta z nowego stosu audio; przy błędzie importu TorchCodec/FFmpeg sprawdź [instrukcję instalacji](https://github.com/pyannote/pyannote-audio#installation). Colab ma zwykle FFmpeg; kod poniżej przekazuje do modelu już zdekodowany sygnał.

In [ ]:
CZY_DIARYZACJA = False
LICZBA_MOWCOW = None

przedzialy_mowcow = []

if CZY_DIARYZACJA:
    from getpass import getpass
    import torch
    from pyannote.audio import Pipeline
    from pyannote.audio.pipelines.utils.hook import ProgressHook
    from faster_whisper.audio import decode_audio

    token_hf = getpass("Token Hugging Face (odczyt): ")
    try:
        diaryzator = Pipeline.from_pretrained(
            "pyannote/speaker-diarization-community-1",
            token=token_hf,
        )
    finally:
        del token_hf

    diaryzator.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    sygnal = decode_audio(PLIK_AUDIO, sampling_rate=16000)
    audio_pamiec = {
        "waveform": torch.from_numpy(sygnal.copy()).unsqueeze(0),
        "sample_rate": 16000,
    }
    parametry = {} if LICZBA_MOWCOW is None else {"num_speakers": LICZBA_MOWCOW}
    with ProgressHook() as postep:
        wynik_diaryzacji = diaryzator(audio_pamiec, hook=postep, **parametry)

    # Wersja exclusive przypisuje w danym momencie najwyżej jednego rozmówcę.
    adnotacja = wynik_diaryzacji.exclusive_speaker_diarization
    for przedzial, _, mowca in adnotacja.itertracks(yield_label=True):
        przedzialy_mowcow.append({
            "start": przedzial.start, "end": przedzial.end, "mowca": mowca
        })
    display(pd.DataFrame(przedzialy_mowcow).head(10))
    del diaryzator, audio_pamiec, sygnal
# Aby wykonać diaryzację, włącz ją w poprzedniej komórce.

### Połącz słowa i rozmówców — GOTOWY KOD

Każdemu słowu przypiszemy rozmówcę o **najdłuższym wspólnym czasie**. Gdy żaden przedział nie pokrywa słowa, zostawiamy `NIEZNANY`. Łączymy kolejne słowa tego samego rozmówcy w wypowiedź.

To przybliżenie: znaczniki ASR mogą być niedokładne. Wersja `exclusive` ułatwia łączenie z tekstem, ale nie zachowuje informacji o jednoczesnym mówieniu kilku osób. Odsłuchaj szczególnie zmiany rozmówcy. Etykiety są ważne tylko w obrębie bieżącego nagrania.

**Do sprawdzenia:** czy osoba wypowiadająca imię jest automatycznie osobą o tym imieniu? Dlaczego nie można ustalić tej relacji wyłącznie na podstawie wykrytego nazwiska?

In [ ]:
# GOTOWY KOD — algorytm niezależny od bibliotek modeli.
def przypisz_mowce(start, end, przedzialy):
    najlepszy_mowca, najdluzszy_czas = "NIEZNANY", 0.0
    for przedzial in przedzialy:
        wspolny_czas = max(
            0.0, min(end, przedzial["end"]) - max(start, przedzial["start"])
        )
        if wspolny_czas > najdluzszy_czas:
            najlepszy_mowca = przedzial["mowca"]
            najdluzszy_czas = wspolny_czas
    return najlepszy_mowca


def polacz_wypowiedzi(slowa, przedzialy):
    wypowiedzi = []
    for slowo in slowa:
        mowca = przypisz_mowce(slowo["start"], slowo["end"], przedzialy)
        if wypowiedzi and wypowiedzi[-1]["mowca"] == mowca:
            wypowiedzi[-1]["tekst"] += slowo["tekst"]
            wypowiedzi[-1]["end"] = slowo["end"]
        else:
            wypowiedzi.append({**slowo, "mowca": mowca})
    for wypowiedz in wypowiedzi:
        wypowiedz["tekst"] = wypowiedz["tekst"].strip()
    return wypowiedzi


wypowiedzi = polacz_wypowiedzi(slowa_asr, przedzialy_mowcow)
if wypowiedzi:
    display(pd.DataFrame(wypowiedzi).head(15))
# Wróć tutaj po wykonaniu transkrypcji, aby otrzymać wypowiedzi.

## 3. Sprawdź jakość transkrypcji: CER — SZABLON

**CER** (*Character Error Rate*) to liczba zamian, usunięć i wstawień znaków podzielona przez liczbę znaków w ręcznej referencji:

`CER = (zamiany + usunięcia + wstawienia) / liczba znaków referencji`

Niższy wynik jest lepszy. `0` oznacza identyczne teksty. Wynik może przekroczyć `1` (100%), gdy model doda wiele znaków. To nie jest procent pewności ani miara poprawności przypisania rozmówców.

1. Odsłuchaj nagranie i samodzielnie zapisz **dokładnie ten sam fragment**, który oceniasz. Dla 30–60 sekund możesz przepisać całość.
2. Wpisz ręczny zapis do `REFERENCJA`. Nie kopiuj wyniku ASR jako referencji.
3. `HIPOTEZA` ma zawierać sam tekst ASR, bez etykiet rozmówców i znaczników czasu.
4. Policz CER surowy i po tej samej normalizacji obu tekstów. Normalizacja poniżej usuwa wielkość liter i interpunkcję oraz ujednolica odstępy; zachowuje polskie znaki i cyfry. Spacje liczą się jako znaki.
5. Poszukaj pomyłek w nazwiskach i numerach. Niski CER całego nagrania może ukryć jeden istotny błąd.

Zapis liczby cyframi i słownie będzie traktowany jako różny. Porównując modele, zachowaj tę samą konwencję referencji i normalizacji.

Źródło API: [jiwer — Character Error Rate](https://jitsi.github.io/jiwer/usage/#character-error-rate).

In [ ]:
# SZABLON — uzupełnij ręczną referencję.
from jiwer import cer

def normalizuj_do_cer(tekst):
    tekst = unicodedata.normalize("NFC", tekst).lower()
    tekst = "".join(
        " " if unicodedata.category(znak).startswith("P") else znak
        for znak in tekst
    )
    return re.sub(r"\s+", " ", tekst).strip()


# GOTOWY PRZYKŁAD: jedna zamiana w trzech znakach.
print(f"Przykład kot → kod: CER = {cer('kot', 'kod'):.1%}")

REFERENCJA = None  # TODO: własny, ręczny zapis tego samego fragmentu audio.
HIPOTEZA = transkrypcja or None  # Dla fragmentu wpisz odpowiadający mu tekst ASR.
wyniki_cer = None

# Uzupełnij REFERENCJA i sprawdź zgodność fragmentów z HIPOTEZA.
if REFERENCJA is not None and HIPOTEZA is not None:
    if not normalizuj_do_cer(REFERENCJA):
        raise ValueError("Referencja po normalizacji jest pusta.")
    wyniki_cer = {
        "CER surowy": cer(REFERENCJA, HIPOTEZA),
        "CER po normalizacji": cer(
            normalizuj_do_cer(REFERENCJA), normalizuj_do_cer(HIPOTEZA)
        ),
    }
    for nazwa, wynik in wyniki_cer.items():
        print(f"{nazwa}: {wynik:.2%}")

## 4. Anonimizacja: wyrażenia regularne

Regex rozpoznaje **wzorzec znaków**, np. cyfry lub adres z symbolem `@`. Nie rozumie tożsamości osoby ani znaczenia wypowiedzi.

Zaczniemy od transkrypcji, a jeżeli jej nie ma — od fikcyjnego tekstu. Przykładowy PESEL jest celowo niepoprawny. Numer telefonu służy wyłącznie jako ciąg testowy, nie do kontaktu.

### Regex na telefon, e-mail i PESEL — SZABLON

Uzupełnij słownik `WZORCE`. Używaj surowych napisów `r"..."`, aby Python nie zmieniał ukośników wzorca.

| Element regexu | Znaczenie |
|---|---|
| `[0-9]` | Jedna cyfra od 0 do 9. |
| `{3}` | Dokładnie trzy powtórzenia. |
| `?` | Poprzedni element może wystąpić zero lub jeden raz. |
| `[ -]?` | Opcjonalna spacja lub myślnik. |
| `\b` | Granica słowa. |
| `(?<![0-9])` / `(?![0-9])` | Przed / po dopasowaniu nie może być kolejnej cyfry. |
| `\+` / `\.` | Dosłowny plus / kropka. |

**Wymagania ćwiczenia:**
- telefon: 9 cyfr, opcjonalny prefiks `+48`, grupy rozdzielone spacjami lub myślnikami;
- e-mail: nazwa użytkownika, `@`, domena i końcówka; wystarczy wzorzec warsztatowy;
- PESEL: dokładnie 11 cyfr, także gdy numer ma niepoprawną sumę kontrolną.

**Próby do sprawdzenia:** telefon `+48 123 456 789`, `123-456-789`, `123456789`; e-mail `anna.kowalska@example.org`; PESEL `12345678901`. Telefon nie powinien dopasowywać dziewięciu cyfr wewnątrz PESEL-u, a PESEL nie powinien dopasowywać części ciągu dwunastocyfrowego.

ASR może zapisać numer słownie, np. „sto dwadzieścia trzy”. Ten prosty regex go nie wykryje. Zaznacz takie przypadki do ręcznego przeglądu.

In [ ]:
# GOTOWY KOD — wybór tekstu roboczego.
TEKST_DEMO = (
    "Anna Kowalska mieszka w Krakowie. Jan Nowak pracuje w Warszawie. "
    "Anna Kowalska podała e-mail anna.kowalska@example.org, "
    "telefon +48 123 456 789 i testowy PESEL 12345678901. "
    "Jan Nowak mówi, że spacer pomaga odpocząć po pracy."
)
tekst_wejsciowy = transkrypcja if transkrypcja.strip() else TEKST_DEMO
print("Źródło:", "własna transkrypcja" if transkrypcja.strip() else "fikcyjny tekst")
print(tekst_wejsciowy)

In [ ]:
# SZABLON — wpisz własne regexy zamiast None.
WZORCE = {
    "TELEFON": None,  # TODO: r"..."
    "EMAIL": None,    # TODO: r"..."
    "PESEL": None,    # TODO: r"..."
}

def wykryj_regex(tekst, wzorce):
    fragmenty = []
    for etykieta, wzorzec in wzorce.items():
        if wzorzec is None:
            continue
        for trafienie in re.finditer(wzorzec, tekst):
            if trafienie.start() == trafienie.end():
                raise ValueError(f"{etykieta}: wzorzec dopasowuje pusty napis.")
            fragmenty.append({
                "start": trafienie.start(), "end": trafienie.end(),
                "etykieta": etykieta, "wartosc": trafienie.group(),
                "priorytet": 2,
            })
    return fragmenty


fragmenty_regex = wykryj_regex(tekst_wejsciowy, WZORCE)
# Uzupełnij trzy wzorce. Pozycje None są pomijane.
display(pd.DataFrame(fragmenty_regex))

### Dla ambitnych: suma kontrolna PESEL — SZABLON

Ostatnia cyfra jest cyfrą kontrolną. Pomnóż pierwsze 10 cyfr przez wagi `1, 3, 7, 9, 1, 3, 7, 9, 1, 3`, zsumuj iloczyny i oblicz `(10 - suma % 10) % 10`. Porównaj wynik z ostatnią cyfrą.

To sprawdzenie nie potwierdza istnienia osoby ani poprawności zakodowanej daty. **Maskuj także numery o złej sumie kontrolnej** — błąd zapisu lub ASR nie usuwa ryzyka identyfikacji. Walidacja służy tu wyłącznie do opisania dopasowania.

Źródło opisu numeru: [Ministerstwo Cyfryzacji — PESEL](https://www.gov.pl/web/cyfryzacja/co-to-jest-numer-pesel-i-jak-sie-go-nadaje).

In [ ]:
# SZABLON — uzupełnij obliczenie cyfry kontrolnej.
def poprawna_suma_pesel(pesel):
    if re.fullmatch(r"[0-9]{11}", pesel) is None:
        return False
    wagi = [1, 3, 7, 9, 1, 3, 7, 9, 1, 3]
    cyfra_kontrolna = None  # TODO: oblicz z pierwszych 10 cyfr i wag.
    if cyfra_kontrolna is None:
        return None  # Nieuzupełnione ćwiczenie.
    return cyfra_kontrolna == int(pesel[-1])


for fragment in fragmenty_regex:
    if fragment["etykieta"] == "PESEL":
        print(fragment["wartosc"], "suma kontrolna:", poprawna_suma_pesel(fragment["wartosc"]))

### Maskowanie tekstu — GOTOWY KOD

Wykrywanie zwraca początek i koniec każdego fragmentu. Zastępujemy fragmenty **od końca tekstu**, żeby wcześniejsze pozycje się nie przesuwały. Nie używamy globalnego `str.replace`, które może zmienić również inny, niewskazany fragment.

Jeśli detektory zgłoszą nakładające się fragmenty, poniższa funkcja wybiera wyższy priorytet, a przy równym priorytecie wcześniejszy i dłuższy fragment. Regexy mają priorytet 2, NER 1. W ten sposób NER nie powinien zasłonić tylko części wykrytego adresu e-mail. To reguła techniczna — odrzucone nakładające się rozpoznania również warto obejrzeć.

In [ ]:
# GOTOWY KOD — te same funkcje wykorzystasz także do wyników NER.
def wybierz_rozlaczne(fragmenty):
    wybrane = []
    kolejnosc = sorted(
        fragmenty,
        key=lambda f: (-f.get("priorytet", 1), f["start"], -(f["end"] - f["start"])),
    )
    for fragment in kolejnosc:
        if not any(
            fragment["start"] < inny["end"] and inny["start"] < fragment["end"]
            for inny in wybrane
        ):
            wybrane.append(fragment)
    return sorted(wybrane, key=lambda f: f["start"])


def maskuj_fragmenty(tekst, fragmenty):
    for fragment in sorted(fragmenty, key=lambda f: f["start"], reverse=True):
        start, end = fragment["start"], fragment["end"]
        if not 0 <= start < end <= len(tekst):
            raise ValueError("Pozycje fragmentu nie pasują do tekstu wejściowego.")
        zamiennik = fragment.get("zamiennik", "[" + fragment["etykieta"] + "]")
        tekst = tekst[:start] + zamiennik + tekst[end:]
    return tekst


wybrane_regex = wybierz_rozlaczne(fragmenty_regex)
tekst_po_regex = maskuj_fragmenty(tekst_wejsciowy, wybrane_regex)
print(tekst_po_regex)
print(f"Wybrano {len(wybrane_regex)} z {len(fragmenty_regex)} dopasowań.")

## 5. Model NER: imiona, nazwiska i miejsca

**NER** (*Named Entity Recognition*) rozpoznaje nazwy z kontekstu. Skorzystamy z [Babelscape/wikineural-multilingual-ner](https://huggingface.co/Babelscape/wikineural-multilingual-ner), którego dane treningowe obejmują również polski. Model zwraca m.in. `PER` (osoby), `LOC` (miejsca) i `ORG` (organizacje). Na warsztacie maskujemy `PER` i `LOC`.

Imię i nazwisko mogą tworzyć jedną encję `PER`. Model może też wykryć je osobno lub pominąć fragment. Wynik `score` nie gwarantuje poprawnego rozpoznania. Odmiana nazw i błędy ASR utrudniają zadanie.

Instalacja `transformers` jest już w sekcji 0. Pierwsze uruchomienie poniższej komórki pobierze model. Licencję i warunki ponownego wykorzystania sprawdź w karcie modelu.

### Instalacja, import i rozpoznanie — GOTOWY KOD

Rozpoznajemy encje w **oryginalnym tekście wejściowym**, tak samo jak regexy. Dzięki temu wszystkie pozycje odnoszą się do tego samego tekstu. `aggregation_strategy="simple"` łączy tokeny w encje, a `stride=64` pozwala analizować dłuższy tekst oknami z zakładką zamiast pomijać jego koniec.

In [ ]:
# GOTOWY KOD — model uruchamiamy na CPU.
from transformers import pipeline

MODEL_NER = "Babelscape/wikineural-multilingual-ner"
ner = pipeline(
    "token-classification",
    model=MODEL_NER,
    tokenizer=MODEL_NER,
    aggregation_strategy="simple",
    stride=64,
    device=-1,
)
encje_ner = ner(tekst_wejsciowy)
fragmenty_ner = [
    {
        "start": int(encja["start"]), "end": int(encja["end"]),
        "etykieta": {"PER": "OSOBA", "LOC": "MIEJSCE"}[encja["entity_group"]],
        "wartosc": tekst_wejsciowy[int(encja["start"]):int(encja["end"])],
        "score": float(encja["score"]), "priorytet": 1,
    }
    for encja in encje_ner
    if encja["entity_group"] in {"PER", "LOC"}
]
fragmenty_wspolne = wybierz_rozlaczne(fragmenty_regex + fragmenty_ner)
display(pd.DataFrame(fragmenty_ner))
print(f"Po połączeniu detektorów: {len(fragmenty_wspolne)} fragmentów.")

### Wariant łatwiejszy: bez rozróżniania osób — WŁASNY KOD

Napisz funkcję, która otrzymuje tekst oraz `fragmenty_wspolne`, a następnie:
- każdą osobę zastępuje przez `[OSOBA]`;
- każde miejsce przez `[MIEJSCE]`;
- telefon, e-mail i PESEL maskuje odpowiednimi etykietami.

Wykorzystaj gotową funkcję `maskuj_fragmenty`: każdy fragment ma już pole `etykieta`.

**Przykład oczekiwanego zachowania:** `Anna Kowalska spotkała Jana Nowaka.` → `[OSOBA] spotkała [OSOBA].` Zależy to od poprawnego wykrycia obu fragmentów.

**Sprawdź ręcznie:** czy zniknęły wszystkie imiona, nazwiska i miejsca, czy zdania pozostały czytelne, czy model nie przeoczył odmiany nazwiska?

In [ ]:
# WŁASNY KOD — napisz ciało funkcji.
def anonimizuj_bez_rozrozniania(tekst, fragmenty):
    # TODO: użyj maskuj_fragmenty i zwróć wynikowy tekst.
    return None


tekst_bez_rozrozniania = anonimizuj_bez_rozrozniania(
    tekst_wejsciowy, fragmenty_wspolne
)
# Uzupełnij funkcję anonimizuj_bez_rozrozniania.
if tekst_bez_rozrozniania is not None:
    print(tekst_bez_rozrozniania)

### Wariant trudniejszy: rozróżnianie osób — WŁASNY KOD

Przygotuj stałe pseudonimy w obrębie jednego tekstu:
- pierwsza osoba → `[OSOBA_1]`, druga → `[OSOBA_2]`;
- kolejne wystąpienie tej samej osoby → ten sam pseudonim;
- miejsca i dane z regexów nadal maskuj etykietami ogólnymi.

**Podpowiedź:** przechodź po fragmentach według `start`. Dla encji `OSOBA` wyznacz klucz na podstawie `wartosc` (`casefold()` i ujednolicenie odstępów), zapamiętaj numer w słowniku i dodaj do kopii fragmentu pole `zamiennik`. Następnie wywołaj `maskuj_fragmenty`.

Powtarzający się identyczny zapis można połączyć automatycznie. „Anna”, „Anny Kowalskiej” i „Anna Kowalska” wymagają dodatkowego rozstrzygnięcia. Opcjonalnie zbuduj słownik `aliasy` po ręcznym sprawdzeniu tekstu. Dwie różne osoby mogą mieć to samo imię i nazwisko — sam zapis nie dowodzi tożsamości. To ćwiczenie nie rozwiązuje automatycznie koreferencji.

Etykieta wykrytej osoby nie oznacza automatycznie rozmówcy z diaryzacji. Ktoś może przecież mówić o innej osobie. Słownika łączącego prawdziwe dane z pseudonimami nie dołączaj do udostępnianego tekstu.

In [ ]:
# WŁASNY KOD — zaimplementuj stałe oznaczenia osób.
def anonimizuj_z_rozroznianiem(tekst, fragmenty, aliasy=None):
    aliasy = {} if aliasy is None else aliasy
    rejestr_osob = {}
    nowe_fragmenty = []
    # TODO:
    # 1. Posortuj fragmenty według początku.
    # 2. Dla OSOBA ustal klucz i nadaj stały numer.
    # 3. Dodaj kopię fragmentu z polem zamiennik do nowe_fragmenty.
    # 4. Zwróć maskuj_fragmenty(tekst, nowe_fragmenty).
    return None


tekst_z_rozroznianiem = anonimizuj_z_rozroznianiem(
    tekst_wejsciowy, fragmenty_wspolne
)
# Uzupełnij funkcję anonimizuj_z_rozroznianiem.
if tekst_z_rozroznianiem is not None:
    print(tekst_z_rozroznianiem)

## 6. Zapisz wynik do pliku tekstowego — GOTOWY KOD

Wybierz własny, obejrzany wynik przez przypisanie go do `tekst_do_zapisu`. Kod zapisuje plik `wyniki/rozmowa_maskowana.txt` w UTF-8, zachowując polskie znaki. Ponowne uruchomienie nadpisuje ten sam plik.

Przed zapisem sprawdź pozostałe wskazówki identyfikujące osobę: adresy, nietypowe stanowisko, nazwę małej organizacji lub opis wyjątkowego zdarzenia. Regexy i model NER nie obejmują wszystkich takich informacji.

Aby pobrać plik w Colabie, ustaw `POBIERZ_PLIK = True`. Zapisywany jest tylko wybrany tekst, bez tabeli wykryć, mapy pseudonimów i oryginalnego nagrania.

In [ ]:
# GOTOWY KOD — wskaż wynik po własnej ocenie.
tekst_do_zapisu = None  # Wybierz: tekst_bez_rozrozniania albo tekst_z_rozroznianiem.
POBIERZ_PLIK = False

# Przypisz wybrany wynik po maskowaniu do tekst_do_zapisu.
if tekst_do_zapisu is not None:
    if not isinstance(tekst_do_zapisu, str) or not tekst_do_zapisu.strip():
        raise ValueError("Wynik musi być niepustym tekstem.")
    katalog_wynikow = Path("wyniki")
    katalog_wynikow.mkdir(exist_ok=True)
    plik_wynikowy = katalog_wynikow / "rozmowa_maskowana.txt"
    plik_wynikowy.write_text(tekst_do_zapisu.rstrip() + "\n", encoding="utf-8")
    print(f"Zapisano: {plik_wynikowy.resolve()}")
    if POBIERZ_PLIK:
        files.download(str(plik_wynikowy))

## 7. Prezentacja wyniku: 3 minuty na parę

Przygotuj krótkie omówienie w notatniku lub na 3 slajdach:

1. **Rozmowa i transkrypcja:** czas nagrania, liczba rozmówców, model ASR, surowy i znormalizowany CER oraz jeden konkretny błąd. Jeśli nie liczyłeś CER, zaznacz to.
2. **Maskowanie:** na fikcyjnym materiale pokaż krótki fragment przed i po maskowaniu. Porównaj regex z NER i warianty oznaczeń osób.
3. **Ograniczenia:** wskaż przeoczoną lub błędnie wykrytą informację, trudność z diaryzacją i przykład danych, których ten proces nie wykrywa. Pokaż zapisany plik.

Nie oceniaj systemu wyłącznie po płynnym brzmieniu tekstu. Błędny numer albo przypisanie słów do innego rozmówcy może mieć znaczenie mimo niskiego CER.

## 8. Sentence Transformer: podobieństwo znaczeń

Sentence Transformer zamienia zdanie na **embedding** — listę liczb opisującą jego znaczenie w przestrzeni modelu. Zdania o podobnym sensie powinny mieć zbliżone reprezentacje, nawet gdy używają innych słów.

Użyjemy [paraphrase-multilingual-MiniLM-L12-v2](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2), modelu wielojęzycznego obsługującego polski. Zwraca wektory o 384 wymiarach. Nie trenujemy nowego modelu: korzystamy z już wyuczonego.

**Podobieństwo kosinusowe** porównuje kierunek wektorów. Dla wektorów o długości 1 jest ich iloczynem skalarnym. Macierz porówna każde zdanie z każdym:
- przekątna ma wartości około `1` — zdanie porównane z samym sobą;
- macierz jest symetryczna;
- większa wartość oznacza większe podobieństwo według tego modelu;
- zakres teoretyczny wynosi od `-1` do `1`; wynik nie jest prawdopodobieństwem;
- wysoki wynik nie dowodzi zgodności poglądów, prawdziwości, pozytywnego nastroju ani tożsamości autora.

**Podobieństwo kosinusowe i korelacja Pearsona to różne miary.** Najpierw pokażemy macierz podobieństwa używaną do porównywania znaczeń, a dodatkowo rzeczywistą macierz korelacji współrzędnych embeddingów. Nie interpretuj tej drugiej jako zależności statystycznej pomiędzy uczestnikami badania.

Źródło: [Sentence Transformers — Semantic Textual Similarity](https://sbert.net/docs/sentence_transformer/usage/semantic_textual_similarity.html).

In [ ]:
# GOTOWY KOD — instalacja sentence-transformers była w sekcji 0.
from sentence_transformers import SentenceTransformer

MODEL_ZDAN = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model_zdan = SentenceTransformer(MODEL_ZDAN, device="cpu")
print("Limit tokenów na tekst:", model_zdan.max_seq_length)

### Zdania i embeddingi — SZABLON z działającym przykładem

Uruchom przykład, a następnie podmień `ZDANIA` na 4–8 własnych krótkich zdań. Uwzględnij:
1. dwie parafrazy tej samej myśli;
2. zdanie o innym temacie;
3. parę z zaprzeczeniem, np. „Lubię…” i „Nie lubię…”.

Przed uruchomieniem przewidź najbardziej podobną parę. Później porównaj swoją ocenę z wynikiem.

`encode(..., normalize_embeddings=True)` tworzy wektory o długości 1. Operator `@` mnoży macierze, a `.T` zamienia wiersze z kolumnami. Dzięki temu `embeddingi @ embeddingi.T` daje podobieństwa wszystkich par.

Model ma limit długości wejścia: długi tekst może zostać obcięty. Tutaj porównujemy krótkie zdania. Dłuższą wypowiedź podziel na krótsze fragmenty przed kodowaniem.

In [ ]:
# SZABLON — zmień zdania, zachowaj co najmniej dwie niepuste pozycje.
ZDANIA = [
    "" # TODO uzupełnić
]

if len(ZDANIA) < 2 or not all(isinstance(s, str) and s.strip() for s in ZDANIA):
    raise ValueError("Podaj przynajmniej dwa niepuste zdania.")

etykiety_zdan = [f"Z{i + 1}" for i in range(len(ZDANIA))]
embeddingi = model_zdan.encode(
    ZDANIA, normalize_embeddings=True, convert_to_numpy=True
)
macierz_podobienstwa = embeddingi @ embeddingi.T
print("Kształt embeddingów (zdania, wymiary):", embeddingi.shape)
display(pd.DataFrame({"etykieta": etykiety_zdan, "zdanie": ZDANIA}))

In [ ]:
# GOTOWY KOD — tabela i mapa cieplna podobieństwa kosinusowego.
tabela_podobienstwa = pd.DataFrame(
    macierz_podobienstwa, index=etykiety_zdan, columns=etykiety_zdan
)
display(tabela_podobienstwa.round(3))
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    tabela_podobienstwa,
    annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, square=True,
    cbar_kws={"label": "Podobieństwo kosinusowe"}, ax=ax,
)
ax.set_title("Podobieństwo znaczeń zdań")
ax.set_xlabel("Zdanie")
ax.set_ylabel("Zdanie")
plt.tight_layout()
plt.show()

### Dodatkowo: wyświetl macierz korelacji Pearsona

W tej macierzy każda para zdań jest porównana poprzez korelację ich 384 współrzędnych. Pearson przed porównaniem odejmuje średnią współrzędnych każdego wektora, dlatego wartości mogą różnić się od podobieństwa kosinusowego.

Przez `embeddingi.T` ustawiamy zdania w kolumnach, a współrzędne w wierszach. Potem `DataFrame.corr()` liczy korelacje między kolumnami. Bez transpozycji otrzymalibyśmy korelacje **wymiarów**, a nie zdań.

Współrzędne modelu nie są niezależnymi pomiarami od ludzi. Ta macierz służy do pokazania różnicy miar, bez testów istotności i wnioskowania o populacji. Dla tego modelu podstawową miarą w ćwiczeniu pozostaje kosinus.

Źródło: [pandas — DataFrame.corr](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html).
Możesz wykonać także dodatkową analizę sentencji.

In [ ]:
# TODO - wykonać macież korelacji